<a href="https://colab.research.google.com/github/isa-pinheiro/Algoritmo-Gale-Shapley/blob/main/transformers_implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import numpy as np

to implement
1. encoder
2. decoder

details
1. word embedding
2. positional encoding
3. multiheaded attention
4. feed foward
5. cross attention

block
1. multihead attention + add & norm
2. masked multihead attention + add & norm
3. feedfoward + add & norm

dropout
- aplicado no final de cada sublayer (antes de ser adicionada ao input e normalizado)
- adicionado ao somatório do embedding e do positional encoding

In [2]:
class InputEmbedding(nn.Module):
    def __init__(self, d_model, vocab_size):
        # d_model tamanho do vetor para cada palavra após o embedding
        # vocab_size tokens com o dataset de sentence pairs
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.embedding = nn.Embedding(self.vocab_size, self.d_model)

    def forward(self, x):
        x = self.embedding(x) # !!não aceita diretamente texto, precisa de que seja transfromado em valores numéricos
        x = x *  np.sqrt(self.d_model)
        return x


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, seq_len, dropout = 0.1):
        # d_model é o tamanho do vetor para cada ser somado a palavra (tamanho embedding)
            # attention is all you need = 512
        # seq_len é o tamanho da sentença do input
            # attention is all you need usou sentence pairs de aproximadamente 25k tokens
        # dropout é a regularização usada
            # p = 0.1 no attention is all you need
        super().__init__()
        self.d_model = d_model
        self.seq_len = seq_len
        self.dropout = nn.Dropout(dropout)

        pe = torch.empty(seq_len, d_model)

        # pos = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1) # (seq_len, 1)
        # pos_exp = pos * torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))

        pos = torch.arange(seq_len).unsqueeze(1)
        pos_exp = pos / torch.pow(10000, 2 * torch.arange(0, d_model, 2) / d_model) # (seq_len, d_model)

        pe[:, 0::2] = torch.sin(pos_exp) # pega somente
        pe[:, 1::2] = torch.cos(pos_exp)

        pe = pe.unsqueeze(0) # (1, seq_len, d_model)

        self.register_buffer('pe', pe)

    def forward(self,x):
        x = x + (self.pe[:, :x.shape[1]])
        x = self.dropout(x)
        return x




In [8]:
# norm_xj = (xj  - mean_j / (sqrt(sqrd(std) + epsilon))) * gamma + beta ; j varia com os batches
# epsilon é usado para estabilizar a normalização, valores não explodirem se o desvio padrão for muito grande
# gamma e beta são valores aprendidos com a backpropagation

class LayerNorm(nn.Module):
    def __init__(self, epsilon = 1e-05):
        super().__init__()
        self.epsilon = epsilon
        self.gamma = nn.Parameter(torch.ones(1))
        self.beta = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        mean = x.mean(dim = -1, keepdim=True)
        std = x.std(dim = -1, keepdim=True)

        x = ((x - mean) / (std + self.epsilon)) * self.gamma + self.beta
        return x


In [15]:
# pointwise feedfoward
# rede feed foward completamente conectada
    # duas transformações lineares com uma relu no meio
# aplica dropout - 0.1
# FFN(x) = max(0, xW1 + b1)W2 + b2
class PointwiseFeedFoward(nn.Module):
    def __init__(self, d_model, dff, dropout = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        self.linear1 = nn.Linear(d_model, dff) # W1, b1
        self.linear2 = nn.Linear(dff, d_model) # W2, b2

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [16]:
d_model = 16
vocab_size = 100
seq_len = 10
dff = 64

x = torch.randint(0, vocab_size, (1, seq_len))
print("input:", x[0][9])

embedding_layer = InputEmbedding(d_model, vocab_size)
x_input_embedded = embedding_layer(x)
print("Saída input embeding: \n ", x_input_embedded[0, 9])

pos_encoding = PositionalEncoding(d_model, seq_len, 0.1)
x_pos_encoded = pos_encoding(x_input_embedded)
print("Saída positional encoding: \n", x_pos_encoded[0, 9])

layernorm = LayerNorm()
x_norm = layernorm(x_pos_encoded)
print("Saída layernorm: \n", x_norm[0,9])

feedforward = PointwiseFeedFoward(d_model, dff)
x_feedforward = feedforward(x_norm)
print("Saída feedforward: \n", x_feedforward[0,9])


input: tensor(92)
Saída input embeding: 
  tensor([ 0.2401,  0.8915,  3.7678, -4.3017,  0.9689,  2.9377, -3.9295,  0.4261,
        -4.3241,  1.6636, -4.1475, -1.4133, -1.5331, -3.7089, -6.1506,  2.1919],
       grad_fn=<SelectBackward0>)
Saída positional encoding: 
 tensor([ 0.7247, -0.0218,  5.0568, -4.0890,  0.0000,  4.3708, -4.3562,  1.5845,
        -0.0000,  2.9595, -4.6082, -0.4593, -1.7034, -3.0099, -6.8340,  0.0000],
       grad_fn=<SelectBackward0>)
Saída layernorm: 
 tensor([ 0.4135,  0.1888,  1.7176, -1.0355,  0.1954,  1.5111, -1.1159,  0.6724,
         0.1954,  1.0863, -1.1918,  0.0571, -0.3174, -0.7107, -1.8618,  0.1954],
       grad_fn=<SelectBackward0>)
Saída feedforward: 
 tensor([-0.0138, -0.3309,  0.7170,  0.0675,  0.0126, -0.1287, -0.2772, -0.3215,
        -0.2093,  0.1663, -0.1463,  0.1705,  0.1174, -0.1436,  0.1453, -0.3937],
       grad_fn=<SelectBackward0>)


In [ ]:
embedding = nn.Embedding(vocab_size, d_model)
print(x.shape)
print(x[0,0])
print(embedding.weight[x[0, 0]])
print(embedding(x)[0,0])